In [1]:
import numpy as np
from helpers import *
from implementations import *

## Import the data

In [2]:
# You have to change the path for it to work
data_path = r'C:\Users\natha\Documents\EPFL\Cours_MA1\ML\ML_course\projects\project1 - withGit\data\dataset'

In [35]:
x_train, x_test, y_train, train_ids, test_ids, headers_train = load_csv_data(data_path, sub_sample=False)

test


In [36]:
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)

(328135, 321)
(109379, 321)
(328135,)


## Convert -1 into 0 in y

In [37]:
y_train[y_train == -1] = 0

# Check if all values are either 0 or 1
if np.all((y_train == 0) | (y_train == 1)):
    print("All values are either 0 or 1.")
else:
    raise ValueError("The array contains values other than 0 or 1.")

All values are either 0 or 1.


In [50]:
x_copy = x_train.copy()
x_copy.shape

(328135, 321)

## Remove columns with too many Nan

In [52]:
# Function to find the count of missing values in each columns
def find_missing_values(data, headers=None):
    num_rows, num_cols = data.shape
    missing_count = np.zeros(num_cols, dtype=int)  
    columns = np.linspace(0,num_cols, num_cols+1)
    
    for col in range(num_cols):
        # Count the missing values
        missing_count[col] = np.sum(np.isnan(data[:, col]))
    if headers : 
        # Returning only the columns with missing values
        missing_info = {headers[col]: missing_count[col] for col in range(num_cols) if missing_count[col] > 0}
    else : 
        missing_info = {columns[col]: missing_count[col] for col in range(num_cols) if missing_count[col] > 0}
    return missing_info

We fix a threshold of 10%

In [56]:
def remove_high_missing_columns(data, headers=None):  
    num_rows, num_cols = data.shape
    threshold = num_rows / 10
    missing_count = find_missing_values(data) 

    # Create a mask for columns to keep
    columns_to_keep = [col for col in range(num_cols) if missing_count.get(col, 0) <= threshold]

    # Filter the data to only include the columns that meet the criteria
    filtered_data = data[:, columns_to_keep]

    # Get the remaining headers
    remaining_headers = [headers[col] for col in columns_to_keep] if headers else None

    return filtered_data, remaining_headers

In [62]:
sliced_x_train, remaining_features = remove_high_missing_columns(x_copy, headers_train)

In [63]:
sliced_x_train.shape

(328135, 139)

In [65]:
print(remaining_features)
print(len(remaining_features))

['Id', '_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'DISPCODE', 'SEQNO', 'HHADULT', 'GENHLTH', 'PHYSHLTH', 'POORHLTH', 'HLTHPLN1', 'PERSDOC2', 'MEDCOST', 'CHECKUP1', 'BPMEDS', 'TOLDHI2', 'CVDSTRK3', 'ASTHNOW', 'CHCSCNCR', 'CHCOCNCR', 'CHCCOPD1', 'HAVARTH3', 'ADDEPEV2', 'CHCKIDNY', 'DIABAGE2', 'SEX', 'MARITAL', 'EDUCA', 'CPDEMO1', 'VETERAN3', 'EMPLOY1', 'CHILDREN', 'INCOME2', 'INTERNET', 'WEIGHT2', 'PREGNANT', 'QLACTLM2', 'USEEQUIP', 'BLIND', 'DECIDE', 'DIFFWALK', 'DIFFDRES', 'DIFFALON', 'LASTSMK2', 'USENOW3', 'MAXDRNKS', 'FRUITJU1', 'FRUIT1', 'FVBEANS', 'FVGREEN', 'FVORANG', 'VEGETAB1', 'EXERHMM2', 'JOINPAIN', 'SEATBELT', 'IMFVPLAC', 'PNEUVAC3', 'ADANXEV', 'QSTVER', 'MSCODE', '_STSTR', '_STRWT', '_RAWRAKE', '_CLLCPWT', '_DUALCOR', '_LLCPWT', '_RFHLTH', '_HCVU651', '_RFHYPE5', '_RFCHOL', '_LTASTH1', '_CASTHM1', '_ASTHMS1', '_DRDXAR1', '_PRACE1', '_MRACE1', '_HISPANC', '_RACE', '_RACEG21', '_RACEGR3', '_RACE_G1', '_AGEG5YR', '_AGE65YR', '_AGE80', '_AGE_G', 'HTIN4', 'HTM4', 'WT

## Remove irrelevant features

In [76]:
def remove_irrelevant_features(data, column_names):
    """
    Removes specific geographic-related features from the dataset using NumPy.

    Parameters:
    data (np.ndarray): The input NumPy array containing BRFSS data.
    column_names (list of str): List of all column names in the dataset.

    Returns:
    np.ndarray: A NumPy array with irrelevant geographic features removed.
    """
    # List of columns to drop based on Category 1
    columns_to_remove = ['Id', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'DISPCODE', 'SEQNO', '_STATE', '_LLCPWT', '_PSU', '_STSTR']
    
    # Find the indices of columns to remove
    indices_to_remove = [column_names.index(col) for col in columns_to_remove if col in column_names]
    
    # Remove the specified columns
    data_filtered = np.delete(data, indices_to_remove, axis=1)
    
    return data_filtered

In [77]:
x_filtered_withoutIrrelevant = remove_irrelevant_features(sliced_x_train, remaining_features)

In [78]:
x_filtered_withoutIrrelevant.shape


(328135, 128)

In [80]:
columns_to_remove = ['Id', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'DISPCODE', 'SEQNO', '_STATE', '_LLCPWT', '_PSU', '_STSTR']
remaining_features_bis = [feature for feature in remaining_features if feature not in columns_to_remove]

In [81]:
print(len(remaining_features_bis))
print(remaining_features_bis)

128
['HHADULT', 'GENHLTH', 'PHYSHLTH', 'POORHLTH', 'HLTHPLN1', 'PERSDOC2', 'MEDCOST', 'CHECKUP1', 'BPMEDS', 'TOLDHI2', 'CVDSTRK3', 'ASTHNOW', 'CHCSCNCR', 'CHCOCNCR', 'CHCCOPD1', 'HAVARTH3', 'ADDEPEV2', 'CHCKIDNY', 'DIABAGE2', 'SEX', 'MARITAL', 'EDUCA', 'CPDEMO1', 'VETERAN3', 'EMPLOY1', 'CHILDREN', 'INCOME2', 'INTERNET', 'WEIGHT2', 'PREGNANT', 'QLACTLM2', 'USEEQUIP', 'BLIND', 'DECIDE', 'DIFFWALK', 'DIFFDRES', 'DIFFALON', 'LASTSMK2', 'USENOW3', 'MAXDRNKS', 'FRUITJU1', 'FRUIT1', 'FVBEANS', 'FVGREEN', 'FVORANG', 'VEGETAB1', 'EXERHMM2', 'JOINPAIN', 'SEATBELT', 'IMFVPLAC', 'PNEUVAC3', 'ADANXEV', 'QSTVER', 'MSCODE', '_STRWT', '_RAWRAKE', '_CLLCPWT', '_DUALCOR', '_RFHLTH', '_HCVU651', '_RFHYPE5', '_RFCHOL', '_LTASTH1', '_CASTHM1', '_ASTHMS1', '_DRDXAR1', '_PRACE1', '_MRACE1', '_HISPANC', '_RACE', '_RACEG21', '_RACEGR3', '_RACE_G1', '_AGEG5YR', '_AGE65YR', '_AGE80', '_AGE_G', 'HTIN4', 'HTM4', 'WTKG3', '_BMI5', '_BMI5CAT', '_RFBMI5', '_CHLDCNT', '_EDUCAG', '_INCOMG', '_SMOKER3', '_RFSMOK3', 'DRN

## Set of rules to improve the dataset: as of now, many values don't make sense

I have read through all the features description in order to select those that were crucial (to keep even if many Nan), those that were meaningless (like the date), and to note down how to convert values that often make no sense into some that can be nicely handled.

In [ ]:
# list of meaningless to add in function
'HHADULT', 'CPDEMO1', 'EMPLOY1', 'CHILDREN', 'INTERNET'

# list of those that could be meaningful but not currently kept (is making sacrifice taking data that has a lot missing. Only make a few exceptions)
'BLOODCHO', 'CHOLCHK', 'CVDINFR4', 'CVDCRHD4', 'ASTHMA3', 'DIABETE3', 'SMOKE100', 'SMOKDAY2', 'STOPSMK2', 'ALCDAY5', 'AVEDRNK2', 'DRNK3GE5', 'EXERANY2'

# Modifs for them if keep them

#DIABETE3 replace 2 by 1, replace 3, 4, 7, 9 by 0
#SMOKDAY2 replace 2 by 1, replace 3, 7, 9 by 0
#USENOW3 replace 2 by 1, replace 3, 7, 9 by 0
#ALCDAY5 replace 777, 888, 999 by 0
#AVEDRNK replace 77, 99 by 0
#DRNK3GE5 replace 77, 88, 99 by 0

# General rule: try to reduce to binary. Keep as it is for kinda continuous data "ex: int from 1 to 9"

#MARITAL if not 1, replace by 0
#PHYSHLTH replace 88, 77, 99 by 0
#POORHLTH replace 88, 77, 99 by 0
#PERSDOC2 replace 2 by 1, replace 3, 7, 9 by 0
#SEX replace 2 by 0
#EDUCA replace 9 by 0
#INCOME2 replace 77 and 99 by 4
#WEIGHT2 if starts by 9, replace by 162
#LASTSMK2 replace 77, 99 by Nan
#MAXDRNKS replace 77, 99 by 0
#FRUITJU1 if >= 300 replace by 0, if < 300 replace by 1
#FRUIT1 if >= 300 replace by 0, if < 300 replace by 1
#FVBEANS if >= 300 replace by 0, if < 300 replace by 1
#FVORANG if >= 300 replace by 0, if < 300 replace by 1
#VEGETAB1 if >= 300 replace by 0, if < 300 replace by 1
#EXERHMM1 replace by 1, 2, 3, 4 if 1<30<100<200

## Deal with missing data (in development)

Input missing data by 0

In [85]:
x_filtered_withoutIrrelevant_0InsteadOfNan = np.nan_to_num(x_filtered_withoutIrrelevant, nan=0.0)
x_filtered_withoutIrrelevant_0InsteadOfNan.shape

(328135, 128)

In [95]:
print(x_filtered_withoutIrrelevant_0InsteadOfNan[:,3])

[1. 1. 1. ... 1. 1. 1.]


In [86]:
column_with_nan = find_missing_values(x_filtered_withoutIrrelevant_0InsteadOfNan)

In [87]:
print(column_with_nan)

{}


## Standardize data to avoid issues with large numbers (is it necessary?? yes for columns with weight for exemple)

In [88]:
def standardize(x):
    """Stadartize the input data x

    Args:
        x: numpy array of shape=(num_samples, num_features)

    Returns:
        standartized data, shape=(num_samples, num_features)
    """
    # ***************************************************
    means=np.mean(x, axis=0)
    stds=np.std(x, axis=0)
    std_data=(x-means)/stds
    # ***************************************************

    return std_data

In [96]:
x_filtered_withoutIrrelevant_0InsteadOfNan_standardized = standardize(x_filtered_withoutIrrelevant_0InsteadOfNan)

# TRAINING THE MODEL

## Split the data

In [90]:
def split_data(x, y, ratio, seed=1):
    """
    split the dataset based on the split ratio. If ratio is 0.8
    you will have 80% of your data set dedicated to training
    and the rest dedicated to testing. If ratio times the number of samples is not round
    you can use np.floor. Also check the documentation for np.random.permutation,
    it could be useful.

    Args:
        x: numpy array of shape (N,), N is the number of samples.
        y: numpy array of shape (N,).
        ratio: scalar in [0,1]
        seed: integer.

    Returns:
        x_tr: numpy array containing the train data.
        x_te: numpy array containing the test data.
        y_tr: numpy array containing the train labels.
        y_te: numpy array containing the test labels.
    """
    
    # set seed
    np.random.seed(seed)
    # ***************************************************
    indices = np.random.permutation(x.shape[0])
    x_shuffled = x[indices]
    y_shuffled = y[indices]
    
    first_index_te = int(np.floor(x.shape[0] * ratio))
    x_tr = x_shuffled[:first_index_te]
    x_te = x_shuffled[first_index_te:]
    y_tr = y_shuffled[:first_index_te]
    y_te = y_shuffled[first_index_te:]
    return x_tr, x_te, y_tr, y_te
    # ***************************************************

In [97]:
x_tr, x_val, y_tr, y_val = split_data(x_filtered_withoutIrrelevant_0InsteadOfNan_standardized, y_train, 0.8, seed=1)
print(x_tr.shape)
print(x_val.shape)
print(y_tr.shape)
print(y_val.shape)

(262508, 128)
(65627, 128)
(262508,)
(65627,)


## Train the model

In [99]:
initial_w = np.zeros(x_tr.shape[1])
max_iters = 10000
gamma = 0.0001

w_opt, loss_opt = logistic_regression(y_tr, x_tr, initial_w, max_iters, gamma)

Current iteration=0, loss=0.6931471805599448
Current iteration=100, loss=0.6925947038487842
Current iteration=200, loss=0.6920599282188102
Current iteration=300, loss=0.6915422444827982
Current iteration=400, loss=0.6910410644036654
Current iteration=500, loss=0.6905558200311775
Current iteration=600, loss=0.6900859630536174
Current iteration=700, loss=0.689630964164642
Current iteration=800, loss=0.6891903124454974
Current iteration=900, loss=0.6887635147626923
Current iteration=1000, loss=0.68835009518118
Current iteration=1100, loss=0.6879495943930531
Current iteration=1200, loss=0.6875615691617035
Current iteration=1300, loss=0.687185591781369
Current iteration=1400, loss=0.686821249551946
Current iteration=1500, loss=0.6864681442689174
Current iteration=1600, loss=0.6861258917282229
Current iteration=1700, loss=0.6857941212458644
Current iteration=1800, loss=0.6854724751920244
Current iteration=1900, loss=0.6851606085394617
Current iteration=2000, loss=0.6848581884259161
Current i

In [100]:
print(w_opt)

[ 4.51604508e-02 -1.70406996e-02 -2.81347876e-03 -2.58712057e-03
 -2.71999712e-03 -3.33887206e-03 -6.64993948e-03 -3.29856145e-02
 -6.22443706e-03 -2.13184700e-02 -5.45967042e-03 -8.83146533e-03
 -8.04292876e-03 -1.61377168e-02 -1.56090671e-02 -6.25271500e-03
 -9.86830489e-03 -2.96297705e-02 -2.07742791e-02 -2.94928578e-03
 -1.14778360e-02  1.93790658e-03 -1.67336283e-02  3.00133967e-02
  1.17304103e-02 -4.64760939e-04  1.62995420e-02 -3.11173443e-03
 -4.45056172e-04 -1.57094444e-02 -1.69118141e-02 -3.17752022e-03
 -3.40792801e-03 -1.73148640e-02 -4.07478052e-03 -6.42792260e-03
 -1.04484684e-02  1.77111918e-03  1.33213222e-02 -8.56414934e-05
  4.24800074e-03  3.00725858e-03  7.11154200e-03  2.39562012e-03
  2.12913780e-03  7.12801056e-03  6.57971293e-03  4.60737380e-03
 -6.08247013e-03 -1.17283293e-02  3.20913807e-03 -6.84392865e-03
 -5.76387457e-04  5.56625623e-04 -1.88024055e-03 -2.82898779e-03
  1.26109386e-03 -3.64398778e-03  2.55057297e-02  2.71834667e-02
 -6.77856507e-03  7.06958

## Apply the model to the validation set

In [101]:
g = x_val @ w_opt
g.shape

(65627,)

In [102]:
limit = 0.5

g[g <= limit] = 0
g[g > limit] = 1

# Check if all values are either 0 or 1
if np.all((g == 0) | (g == 1)):
    print("All values are either 0 or 1.")
else:
    raise ValueError("The array contains values other than 0 or 1.")

All values are either 0 or 1.


## Check our accuracy and F1

In [103]:
accuracy = np.mean(y_val == g)

print("Accuracy:", accuracy)

Accuracy: 0.9036524601155013


In [104]:
TP = np.sum((y_val == 1) & (g == 1))
FP = np.sum((y_val == 0) & (g == 1))
FN = np.sum((y_val == 1) & (g == 0))

# Calculate Precision and Recall
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0

# Calculate F1 Score
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1_score)

Precision: 0.39945155393053017
Recall: 0.2315027370651598
F1 Score: 0.29312465064281723
